# IncomeETF Pitch EDA: does dividend-capture demand distort income ETFs around ex-dates?

**The mechanism.** Covered-call and "income" ETFs (JEPI, QYLD, and especially the YieldMax single-stock funds, paying
1–3% *per distribution*) are bought by retail investors for the payout. Buyers who want the distribution bid the fund
up into the last cum day and sell after it, so the price should fall by *more* than the distribution on the ex-date
(total return below what the underlying implies).

**Design.** Abnormal total return = ETF log return (adjusted for distributions) minus a rolling 120-day beta × the
underlying's return (SPY/QQQ/IWM, or the single stock for YieldMax). Windows: 3 days before the ex-date, the ex-date,
and 3 days after. Data: Yahoo (`collect.py`). TSLA prices failed to download, so TSLY drops out.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
P = pd.read_csv("data/prices.csv", parse_dates=["date"]); A = P.pivot(index="date", columns="ticker", values="adj_close"); C = P.pivot(index="date", columns="ticker", values="close")
dv = pd.read_csv("data/dividends.csv", parse_dates=["ex_date"]); pairs = pd.read_csv("data/pairs.csv").set_index("etf").under
R = np.log(A).diff() * 1e4; cal = R.index
rows = []
for t, u in pairs.items():
    if t not in R or u not in R: continue
    x, y = R[t], R[u]
    beta = x.rolling(120, min_periods=60).cov(y) / y.rolling(120, min_periods=60).var()
    ab = x - beta.shift(1) * y
    for r in dv[dv.ticker == t].itertuples():
        i = cal.searchsorted(r.ex_date)
        if i < 130 or i + 3 >= len(cal) or cal[i] != r.ex_date: continue
        rows.append({"etf": t, "date": cal[i], "yield_bp": r.amount / C[t].iloc[i - 1] * 1e4, "pre3": ab.iloc[i - 3:i].sum(), "ex": ab.iloc[i], "post3": ab.iloc[i + 1:i + 4].sum()})
E = pd.DataFrame(rows)
E["ym"] = E.etf.isin(["TSLY", "NVDY", "CONY", "MSTY", "AMDY", "APLY", "AMZY", "GOOY", "FBY", "NFLY", "MSFO", "PLTY", "ULTY", "YMAX"])
for g, m in [("index income ETFs", ~E.ym), ("YieldMax single-stock", E.ym)]:
    d = E[m]
    print(g, "n", len(d), f"first {d.date.min().date()}", "median distribution", f"{d.yield_bp.median():.0f}bp", {c: f"{d[c].mean():+.1f} (t {d[c].mean() / d[c].sem():+.2f})" for c in ["pre3", "ex", "post3"]})
    d2 = d[d.yield_bp > d.yield_bp.quantile(0.75)]
    print("   top-quartile distributions:", {c: f"{d2[c].mean():+.1f} (t {d2[c].mean() / d2[c].sem():+.2f})" for c in ["pre3", "ex", "post3"]})

index income ETFs n 709 first 2013-07-24 median distribution 88bp {'pre3': '+0.1 (t +0.03)', 'ex': '-2.5 (t -1.81)', 'post3': '-4.2 (t -1.55)'}
   top-quartile distributions: {'pre3': '-0.5 (t -0.10)', 'ex': '-0.0 (t -0.01)', 'post3': '-5.8 (t -0.84)'}
YieldMax single-stock n 1054 first 2023-06-07 median distribution 125bp {'pre3': '-4.5 (t -1.70)', 'ex': '-6.4 (t -3.37)', 'post3': '+0.4 (t +0.15)'}
   top-quartile distributions: {'pre3': '-11.8 (t -1.91)', 'ex': '-17.4 (t -3.70)', 'post3': '+1.5 (t +0.26)'}


**Interpretation.**
- **Index income ETFs: nothing** (ex-date −2.5bp, t −1.8; no pre-ex run-up).
- **YieldMax: the price over-drops on the ex-date:** −6bp abnormal (t −3.4), −17bp for the largest distributions
  (t −3.7). There is no run-up before (−4.5bp), so it is ex-day selling (capture sellers) more than a pre-ex bid.
- **Not tradable.** Capturing it means shorting at the cum-day close and covering on the ex-date: 10–15bp of spread
  on these funds plus borrow (often expensive) is about the size of the edge. The sample only starts in 2023, so there
  is no out-of-sample period.

**Verdict: real but too small and too young.**